# 12주 · 축차법·차원 해석 도입

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 직접축차법·표준축차법 / 차원 해석 도입

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week12/CE272_W12.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week12) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 각 S절 개별 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: 직접축차·표준축차 첫 구간

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

## S01. 원본 조건의 직접축차법

**슬라이드 연결**: CE272_W12a_Steady_Nonuniform_Flow_5.pptx p3, 4, 5, 6, 7

**자료 구분**: 원본 Step Method Excel

**먼저 예측**: 등류수심을 넘어 이어 붙인 계산점도 같은 M1 수면곡선인가?

**문제**: Q=30 m³/s, b=10 m, z=2, n=0.013, S0=0.001, 하류 수심 5 m를 사용하시오. 상류 방향을 양의 거리로 정의하고 직접축차법의 첫 구간을 손계산하시오. 등류수심에 접근할 때 거리 간격의 변화를 설명하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
yn=hyd.normal_depth(30,10,2,.013,.001)
yc=hyd.critical_depth(30,10,2)
ys=np.round(np.arange(5.,1.09,-.1),10)  # 마지막 수심 1.10 m: yn보다 위에서 중단
direct=hyd.direct_step(ys)
display(direct.head());display(direct.tail())
print('yn, yc:',yn,yc,'; max energy residual:',direct['residual'].abs().max())
plt.plot(direct.x_up,direct.y);plt.axhline(yn,ls='--');plt.xlabel('upstream distance (m)');plt.ylabel('y (m)');plt.grid();plt.show()

**검토 문제**: 원본의 표시값 yn=1.09 m와 정밀 계산값을 구분하시오. S0−Sf가 0에 접근하거나 부호가 바뀌는 구간을 연결하면 생기는 문제를 설명하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## S02. 표준축차법의 셀 참조·스텝 검토

**슬라이드 연결**: CE272_W12a_Steady_Nonuniform_Flow_5.pptx p8, 9, 10, 11

**자료 구분**: 원본 Standard 시트 대조·재계산

**먼저 예측**: Excel 목표 잔차가 작으면 올바른 두 단면을 사용했다고 보장되는가?

**문제**: 같은 원본 조건으로 첫 1,000 m의 표준축차 해를 구하시오. 원본 Standard 시트 K6:K10·N6:N10의 인접 단면 참조를 검토하시오. 올바른 에너지식의 거리 스텝을 250·100·50 m로 줄여 4,500 m 지점의 수심을 비교하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
first=hyd.standard_step([0,1000])
display(first)
rows=[]
for dx in [250,100,50]:
    trace=hyd.standard_step(np.arange(0,4500+dx,dx))
    rows.append({'dx_m':dx,'y_at_4500_m':trace.iloc[-1].y,
                 'max_energy_residual':trace.residual.abs().max()})
display(pd.DataFrame(rows))
try:
    hyd.standard_step([0,1000,1500,2000,2500,3000,3500,4000,4500])
except ValueError as error:
    print('Coarse step review:',error)

**검토 문제**: 작은 에너지 잔차와 작은 이산화 오차의 차이를 설명하시오. 식의 셀 참조 오류와 물리적 가지를 벗어나는 수치 스텝 문제를 구분하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## S03. 차원 동차성과 Buckingham π

**슬라이드 연결**: CE272_W12b_Dimensional_Analysis_1.pptx p15, 16, 17, 18, 19, 20, 21, 22

**자료 구분**: 원본 원리·차원표 재구성

**먼저 예측**: 변수 5개에서 기본 차원 3개를 빼면 언제나 정확히 2개인가?

**문제**: 압력경사, ρ, μ, V, D의 M·L·T 차원행렬을 작성하시오. 행렬의 독립 차원 수를 확인하고 원본의 두 무차원군이 차원 동차성을 만족하는지 검산하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
# 변수 순서: pressure gradient Δp/L, rho, mu, V, D
A=np.array([[1,1,1,0,0],[-2,-3,-1,1,1],[-2,0,-1,-1,0]])
pi1=np.array([1,-1,0,-2,1])
pi2=np.array([0,1,-1,1,1])
display(pd.DataFrame(A,index=['M','L','T'],columns=['dp_per_L','rho','mu','V','D']))
print('rank, independent pi terms:',np.linalg.matrix_rank(A),A.shape[1]-np.linalg.matrix_rank(A))
print('dimension checks:',A@pi1,A@pi2)

**검토 문제**: 기본 단위의 개수와 차원행렬의 독립 행 수를 구분하시오. 무차원군만으로 미지 함수의 형태와 계수가 자동 결정되는지도 설명하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## 학습 기록

좌표 방향·인접 단면 참조·근의 가지·스텝 크기를 근거로 수면곡선을 검토하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록

In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')